# Mirage · seed 1, one arm per run

Trains ONE GRPO arm with seed 1 (300 steps, ~4 h on a T4), reusing seed 0's suites and pass-rate filter,
so the only change from seed 0 is the seed. Set `ARM` below: `"single"` first, `"suite"` in the next run.

**Inputs:** `spider` (Your Datasets) + `notebook376c8ebfe2` (Your Work: seed 0's suites + filter).
For the second arm also add this notebook's own previous version (so both seed-1 arms end up in one output).
Accelerator **GPU T4 x2**, Internet **On** → **Save Version → Save & Run All (Commit)**.

In [ ]:
ARM = "single"          # <- "single" now; "suite" in the next run
SEED, STEPS, BUDGET_H = 1, 300, 5.5    # identical for both seed-1 arms, so their manifests match

In [ ]:
# --- setup ---------------------------------------------------------------
import os, subprocess, sys, time, json, shutil, glob
T0 = time.time()
SPIDER_DIR = "/kaggle/input/datasets/siddhartharajat/spider/spider_data"
RUNS = "/kaggle/working/runs"
os.environ.update(SPIDER_DIR=SPIDER_DIR, MIRAGE_RUNS=RUNS, CUDA_VISIBLE_DEVICES="0",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
os.makedirs(RUNS, exist_ok=True)
os.chdir("/kaggle/working")
if not os.path.exists("mirage"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SiddharthaRaja/Mirage", "mirage"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "mirage[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
os.chdir("/kaggle/working/mirage")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{SPIDER_DIR}/train_spider.json"), "SPIDER_DIR is wrong"
import torch
assert torch.cuda.is_available(), "NO GPU: set Accelerator to GPU T4 x2"
print(torch.cuda.get_device_name(0), "| arm:", ARM, "| seed:", SEED)

In [ ]:
# --- bring in seed 0's suites + filter (and any earlier seed-1 arms) -------------
runs_dirs = []
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if root.endswith("/runs"):
        runs_dirs.append(root)
    if root.count("/") - "/kaggle/input".count("/") > 6:
        dirs[:] = []
print("runs folders found:", runs_dirs)
for src in runs_dirs:
    for part in ["suites", "filter"]:
        if os.path.isdir(f"{src}/{part}") and not os.path.isdir(f"{RUNS}/{part}"):
            shutil.copytree(f"{src}/{part}", f"{RUNS}/{part}")
    for d in glob.glob(f"{src}/arms/*-s1"):                 # earlier seed-1 arm(s): keep them in this output
        shutil.copytree(d, f"{RUNS}/arms/{os.path.basename(d)}", dirs_exist_ok=True)
assert os.path.exists(f"{RUNS}/filter/keep.jsonl"), "seed 0's filter not found: Add Input -> Your Work -> notebook376c8ebfe2"
assert len(os.listdir(f"{RUNS}/suites")) >= 160, "seed 0's suites not found"
print("suites:", len(os.listdir(f"{RUNS}/suites")), "| filter kept:",
      sum(json.loads(l)["keep"] for l in open(f"{RUNS}/filter/keep.jsonl")),
      "| seed-1 arms present:", sorted(os.listdir(f"{RUNS}/arms")) if os.path.isdir(f"{RUNS}/arms") else [])

In [ ]:
# --- train ------------------------------------------------------------------
OUT = f"{RUNS}/arms/{ARM}-s1"
cmd = (f"mirage grpo -c configs/grpo_{ARM}.yaml --set out={OUT} --set grpo.seed={SEED} "
       f"--set grpo.max_steps={STEPS} --set budget.max_hours={BUDGET_H}")
print("$", cmd, flush=True)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
log = []
for line in p.stdout:
    print(line, end="", flush=True); log.append(line)
rc = p.wait()
os.makedirs(f"{RUNS}/logs", exist_ok=True)
open(f"{RUNS}/logs/grpo_{ARM}-s1.log", "w").write("".join(log))
print("exit code", rc, "| hours:", round((time.time() - T0) / 3600, 2))

In [ ]:
# --- audit + summary ------------------------------------------------------
out = subprocess.run(f"mirage audit {OUT}/ledger.jsonl", shell=True, capture_output=True, text=True).stdout
open(f"{RUNS}/logs/audit_{ARM}-s1.log", "w").write(out)
cks = sorted(glob.glob(f"{OUT}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
summary = f"""# Seed 1 · {ARM} arm
steps reached: {cks[-1].rsplit('-', 1)[1] if cks else 'none'} / {STEPS}
adapter saved: {os.path.exists(f'{OUT}/adapter/adapter_config.json')}
hours: {(time.time() - T0) / 3600:.2f}

{out}"""
open(f"{RUNS}/SEED1_{ARM}.md", "w").write(summary)
print(summary)